In [2]:
import pandas as pd
import numpy as np
import random
import gc
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import precision_score, recall_score

In [3]:
filenames = range(20251101,20251127)
# filenames = range(20251120,20251127)

In [4]:
feature_cols = pd.read_parquet(f"../datasrc/data/features/20251101.parquet").columns.tolist()
label_cols = pd.read_parquet(f"../datasrc/data/labels/prepaid_to_postpaid_20251101.parquet").columns.tolist()

all_cols = feature_cols + [label_cols[2]]


In [5]:

all_data_chunks = []
combined_df = pd.DataFrame(columns = all_cols)



In [6]:

for filename in filenames:
    feature_df = pd.read_parquet(f"../datasrc/data/features/{filename}.parquet")
    label_df = pd.read_parquet(f"../datasrc/data/labels/prepaid_to_postpaid_{filename}.parquet")

    


    left_cols = feature_df.columns[:2].tolist()#joining on sub_id,ctx_date
    right_cols = label_df.columns[:2].tolist()

    joined_df = pd.merge(feature_df, label_df, left_on=left_cols, right_on=right_cols)
    all_data_chunks.append(joined_df)
    del feature_df
    del label_df
    gc.collect()


    
combined_df = pd.concat(all_data_chunks, axis=0,ignore_index=True)
del all_data_chunks
gc.collect()

0

In [7]:
# combined_df["Subs_Id"] = combined_df["Subs_Id"].astype(str)

In [8]:
# combined_df.shape

In [9]:
# combined_df.head

In [10]:
combined_df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5200000 entries, 0 to 5199999
Columns: 110 entries, Subs_Id to IsMigrated
dtypes: datetime64[us](21), float32(4), float64(46), int32(3), int64(23), object(4), uint32(9)
memory usage: 4.7 GB


In [11]:
object_cols = combined_df.select_dtypes(include=['object']).columns.tolist()
print(f"Total object columns: {len(object_cols)}")
print(object_cols)

Total object columns: 4
['ContextDate', 'LastAddOnPackage', 'LastMacroAddOnPackage', 'LastMicroAddOnPackage']


In [12]:

all_accs = set()
with open("../all_subs_id/subs_id.csv") as mysrc:
    for line in mysrc:
        sub_id = int(line.strip())
        all_accs.add(sub_id)
acc_list = list(all_accs)
random.shuffle(acc_list)
split_idx = int(len(acc_list) * 0.8)
set_80 = set(acc_list[:split_idx])
set_20 = set(acc_list[split_idx:])
print(len(set_80))
print(len(set_20))

160000
40000


In [13]:
model_train = combined_df[combined_df["Subs_Id"].isin(set_80)]

model_test = combined_df[combined_df["Subs_Id"].isin(set_20)]



print(f" rows in original DF: {len(combined_df)}")
del combined_df
gc.collect()
print(f"rows in train DF: {len(model_train)}")
print(f"rows in test DF: {len(model_test)}")

 rows in original DF: 5200000
rows in train DF: 4160000
rows in test DF: 1040000


In [14]:
# model_train.info(memory_usage='deep')

In [15]:
# model_test.info(memory_usage='deep')

In [16]:
label_train = model_train['IsMigrated']
label_test = model_test['IsMigrated']

In [17]:
label_train.value_counts()

IsMigrated
0    4152106
1       7894
Name: count, dtype: int64

In [18]:
label_test.value_counts()

IsMigrated
0    1037887
1       2113
Name: count, dtype: int64

In [19]:
model_train = model_train.drop(columns = ['Subs_Id','Cust_Id','ContextDate','IsMigrated','LastAddOnPackage', 'LastMacroAddOnPackage', 'LastMicroAddOnPackage'])
model_test =  model_test.drop(columns = ['Subs_Id','Cust_Id','ContextDate','IsMigrated','LastAddOnPackage', 'LastMacroAddOnPackage', 'LastMicroAddOnPackage'])



In [20]:
print(model_train.dtypes.to_string())
# print(model_test.dtypes.to_string())

TotalVoiceChargedDuration60                    float64
TotalVoiceFreeDuration60                       float64
TotalDataChargeUsage60                         float64
TotalDataFreeUsage60                           float64
TotalNetwork4gUsage60                          float64
TotalNetwork5gUsage60                          float64
TotalSmsFreeCount60                            float64
TotalVoiceChargeAmount60                       float64
TotalDataRoamingFreeUsage60                    float64
TotalDataRoamingChargeUsage60                  float64
TotalVoiceChargedDuration30                    float64
TotalVoiceFreeDuration30                       float64
TotalDataChargeUsage30                         float64
TotalDataFreeUsage30                           float64
TotalNetwork4gUsage30                          float64
TotalNetwork5gUsage30                          float64
TotalSmsFreeCount30                            float64
TotalVoiceChargeAmount30                       float64
LastVoiceC

In [21]:
date_columns = ["LastVoiceChargeUsageDate","LastVoiceFreeUsageDate", "LastVoiceRoamingChargeUsageDate","LastVoiceRoamingFreeUsageDate",
"LastSMSChargeUsageDate","LastSMSFreeUsageDate","LastSMSRoamingChargeUsageDate","LastSMSRoamingFreeUsageDate",
"LastDataFreeUsageDate", "LastDataChargeUsageDate","LastDataRoamingFreeUsageDate", "LastDataRoamingChargeUsageDate",
"LastNetwork4gUsageDate","LastNetwork5gUsageDate","LastRechargeDate","MacroAddonExpiryDate","MicroAddonExpiryDate",
"LastAddOnPurchaseDate","LastMacroAddOnPurchaseDate","LastMicroAddOnPurchaseDate","LastPromotionPurchaseDate",
"LastVoiceChargeUsageDate","LastVoiceFreeUsageDate","LastVoiceRoamingChargeUsageDate","LastVoiceRoamingFreeUsageDate",
"LastSMSChargeUsageDate","LastSMSFreeUsageDate","LastSMSRoamingChargeUsageDate","LastSMSRoamingFreeUsageDate",
"LastDataFreeUsageDate","LastDataChargeUsageDate","LastDataRoamingFreeUsageDate","LastDataRoamingChargeUsageDate",
"LastNetwork4gUsageDate","LastNetwork5gUsageDate","LastRechargeDate","MacroAddonExpiryDate","MicroAddonExpiryDate",
"LastAddOnPurchaseDate","LastMacroAddOnPurchaseDate","LastMicroAddOnPurchaseDate","LastPromotionPurchaseDate"]

In [22]:
#dropping date columns

model_train = model_train.drop(columns = date_columns)

model_test =  model_test.drop(columns = date_columns)


In [23]:
model_train = model_train.apply(pd.to_numeric, errors='coerce')
model_test = model_test.apply(pd.to_numeric, errors='coerce')

model_train = model_train.fillna(0)
model_test = model_test.fillna(0)

In [24]:


model_train['VoiceChargedDurationVelocity730'] = model_train['TotalVoiceChargedDuration7']/(model_train['TotalVoiceChargedDuration30']/4+0.01)
model_train['VoiceFreeDurationVelocity730'] = model_train['TotalVoiceFreeDuration7']/(model_train['TotalVoiceFreeDuration30']/4+0.01)

model_train['DataChargeVelocity730'] = model_train['TotalDataChargeUsage7']/(model_train['TotalDataChargeUsage30']/4+0.01)
model_train['DataFreeVelocity730'] = model_train['TotalDataFreeUsage7']/(model_train['TotalDataFreeUsage30']/4+0.01)

model_train['Network4gUsageVelocity730'] = model_train['TotalNetwork4gUsage7']/(model_train['TotalNetwork4gUsage30']/4+0.01)
model_train['Network5gUsageVelocity730'] = model_train['TotalNetwork5gUsage7']/(model_train['TotalNetwork5gUsage30']/4+0.01)

model_train['SmsFreeCountVelocity730'] = model_train['TotalSmsFreeCount7']/(model_train['TotalSmsFreeCount30']/4+0.01)

model_train['VoiceChargeAmountVelocity730'] = model_train['TotalVoiceChargeAmount7']/(model_train['TotalVoiceChargeAmount30']/4+0.01)
 
model_train['RechargeAmountVelocity730'] = model_train['TotalRechargeAmount7']/(model_train['TotalRechargeAmount30']/4+0.01)

model_train['AverageBalanceVelocity730'] = model_train['AverageBalance7']/(model_train['AverageBalance30']/4+0.01)

model_train['MacroAddOnCountVelocity1530'] = model_train['MacroAddOnCount15']/(model_train['MacroAddOnCount30']/2+0.01)
model_train['MacroAddOnPriceVelocity1530'] = model_train['MacroAddOnPrice15']/(model_train['MacroAddOnPrice30']/2+0.01)

model_train['MicroAddOnCountVelocity1530'] = model_train['MicroAddOnCount15']/(model_train['MicroAddOnCount30']/2+0.01)
model_train['MicroAddOnPriceVelocity1530'] = model_train['MicroAddOnPrice15']/(model_train['MicroAddOnPrice30']/2+0.01)








In [25]:


model_test['VoiceChargedDurationVelocity730'] = model_test['TotalVoiceChargedDuration7']/(model_test['TotalVoiceChargedDuration30']/4+0.01)
model_test['VoiceFreeDurationVelocity730'] = model_test['TotalVoiceFreeDuration7']/(model_test['TotalVoiceFreeDuration30']/4+0.01)

model_test['DataChargeVelocity730'] = model_test['TotalDataChargeUsage7']/(model_test['TotalDataChargeUsage30']/4+0.01)
model_test['DataFreeVelocity730'] = model_test['TotalDataFreeUsage7']/(model_test['TotalDataFreeUsage30']/4+0.01)

model_test['Network4gUsageVelocity730'] = model_test['TotalNetwork4gUsage7']/(model_test['TotalNetwork4gUsage30']/4+0.01)
model_test['Network5gUsageVelocity730'] = model_test['TotalNetwork5gUsage7']/(model_test['TotalNetwork5gUsage30']/4+0.01)

model_test['SmsFreeCountVelocity730'] = model_test['TotalSmsFreeCount7']/(model_test['TotalSmsFreeCount30']/4+0.01)

model_test['VoiceChargeAmountVelocity730'] = model_test['TotalVoiceChargeAmount7']/(model_test['TotalVoiceChargeAmount30']/4+0.01)
 
model_test['RechargeAmountVelocity730'] = model_test['TotalRechargeAmount7']/(model_test['TotalRechargeAmount30']/4+0.01)

model_test['AverageBalanceVelocity730'] = model_test['AverageBalance7']/(model_test['AverageBalance30']/4+0.01)

model_test['MacroAddOnCountVelocity1530'] = model_test['MacroAddOnCount15']/(model_test['MacroAddOnCount30']/2+0.01)
model_test['MacroAddOnPriceVelocity1530'] = model_test['MacroAddOnPrice15']/(model_test['MacroAddOnPrice30']/2+0.01)

model_test['MicroAddOnCountVelocity1530'] = model_test['MicroAddOnCount15']/(model_test['MicroAddOnCount30']/2+0.01)
model_test['MicroAddOnPriceVelocity1530'] = model_test['MicroAddOnPrice15']/(model_test['MicroAddOnPrice30']/2+0.01)








In [27]:


# model_train['VoiceChargedDurationVelocity760'] = model_train['TotalVoiceChargedDuration7']/(model_train['TotalVoiceChargedDuration60']/8.5+0.01)
# model_train['VoiceFreeDurationVelocity760'] = model_train['TotalVoiceFreeDuration7']/(model_train['TotalVoiceFreeDuration60']/8.5+0.01)

# model_train['DataChargeVelocity760'] = model_train['TotalDataChargeUsage7']/(model_train['TotalDataChargeUsage60']/8.5+0.01)
# model_train['DataFreeVelocity760'] = model_train['TotalDataFreeUsage7']/(model_train['TotalDataFreeUsage60']/8.5+0.01)

# model_train['Network4gUsageVelocity760'] = model_train['TotalNetwork4gUsage7']/(model_train['TotalNetwork4gUsage60']/8.5+0.01)
# model_train['Network5gUsageVelocity760'] = model_train['TotalNetwork5gUsage7']/(model_train['TotalNetwork5gUsage60']/8.5+0.01)

# model_train['SmsFreeCountVelocity760'] = model_train['TotalSmsFreeCount7']/(model_train['TotalSmsFreeCount60']/8.5+0.01)

# model_train['VoiceChargeAmountVelocity760'] = model_train['TotalVoiceChargeAmount7']/(model_train['TotalVoiceChargeAmount60']/8.5+0.01)
 
# model_train['RechargeAmountVelocity760'] = model_train['TotalRechargeAmount7']/(model_train['TotalRechargeAmount60']/8.5+0.01)

# model_train['AverageBalanceVelocity730'] = model_train['AverageBalance7']/(model_train['AverageBalance30']/4+0.01)

# model_train['MacroAddOnCountVelocity1530'] = model_train['MacroAddOnCount15']/(model_train['MacroAddOnCount30']/2+0.01)
# model_train['MacroAddOnPriceVelocity1530'] = model_train['MacroAddOnPrice15']/(model_train['MacroAddOnPrice30']/2+0.01)

# model_train['MicroAddOnCountVelocity1530'] = model_train['MicroAddOnCount15']/(model_train['MicroAddOnCount30']/2+0.01)
# model_train['MicroAddOnPriceVelocity1530'] = model_train['MicroAddOnPrice15']/(model_train['MicroAddOnPrice30']/2+0.01)










In [28]:


# model_test['VoiceChargedDurationVelocity760'] = model_test['TotalVoiceChargedDuration7']/(model_test['TotalVoiceChargedDuration60']/8.5+0.01)
# model_test['VoiceFreeDurationVelocity760'] = model_test['TotalVoiceFreeDuration7']/(model_test['TotalVoiceFreeDuration60']/8.5+0.01)

# model_test['DataChargeVelocity760'] = model_test['TotalDataChargeUsage7']/(model_test['TotalDataChargeUsage60']/8.5+0.01)
# model_test['DataFreeVelocity760'] = model_test['TotalDataFreeUsage7']/(model_test['TotalDataFreeUsage60']/8.5+0.01)

# model_test['Network4gUsageVelocity760'] = model_test['TotalNetwork4gUsage7']/(model_test['TotalNetwork4gUsage60']/8.5+0.01)
# model_test['Network5gUsageVelocity760'] = model_test['TotalNetwork5gUsage7']/(model_test['TotalNetwork5gUsage60']/8.5+0.01)

# model_test['SmsFreeCountVelocity760'] = model_test['TotalSmsFreeCount7']/(model_test['TotalSmsFreeCount60']/8.5+0.01)

# model_test['VoiceChargeAmountVelocity760'] = model_test['TotalVoiceChargeAmount7']/(model_test['TotalVoiceChargeAmount60']/8.5+0.01)
 
# model_test['RechargeAmountVelocity760'] = model_test['TotalRechargeAmount7']/(model_test['TotalRechargeAmount60']/8.5+0.01)

# model_test['AverageBalanceVelocity730'] = model_test['AverageBalance7']/(model_test['AverageBalance30']/4+0.01)

# model_test['MacroAddOnCountVelocity1530'] = model_test['MacroAddOnCount15']/(model_test['MacroAddOnCount30']/2+0.01)
# model_test['MacroAddOnPriceVelocity1530'] = model_test['MacroAddOnPrice15']/(model_test['MacroAddOnPrice30']/2+0.01)

# model_test['MicroAddOnCountVelocity1530'] = model_test['MicroAddOnCount15']/(model_test['MicroAddOnCount30']/2+0.01)
# model_test['MicroAddOnPriceVelocity1530'] = model_test['MicroAddOnPrice15']/(model_test['MicroAddOnPrice30']/2+0.01)










In [38]:

features_to_drop = ['TotalVoiceChargedDuration60','TotalVoiceChargedDuration14','TotalVoiceFreeDuration60','TotalVoiceFreeDuration14',
                    'TotalDataChargeUsage60','TotalDataChargeUsage14','TotalDataFreeUsage60','TotalDataFreeUsage14',
                    'TotalNetwork4gUsage60','TotalNetwork4gUsage14','TotalNetwork5gUsage60','TotalNetwork5gUsage14','TotalSmsFreeCount60',
                    'TotalSmsFreeCount14','TotalVoiceChargeAmount60','TotalVoiceChargeAmount14','TotalRechargeAmount60',
                    'TotalRechargeAmount14']

In [39]:
model_train = model_train.drop(columns = features_to_drop)

model_test =  model_test.drop(columns = features_to_drop)


In [40]:
# idx_pos = label_train[label_train == 1].index
# idx_neg = label_train[label_train == 0].index

In [41]:
# n_neg_samples = len(idx_pos) * 5
# idx_neg_sampled = random.sample(list(idx_neg), n_neg_samples)
# balanced_idx = list(idx_pos) + idx_neg_sampled
# random.shuffle(balanced_idx)

In [42]:
# model_train_resampled = model_train.loc[balanced_idx]
# label_train_resampled = label_train.loc[balanced_idx]

In [43]:

notmigrated_count = (label_train == 0).sum()
migrated_count = (label_train == 1).sum()
weight_ratio = notmigrated_count / migrated_count

cat_model = CatBoostClassifier(
    iterations=500,            
    learning_rate=0.1,
    depth=6,                  
    l2_leaf_reg=3,            
    border_count=254,
    scale_pos_weight=weight_ratio, # <imp>
   
    verbose=False,        
    random_seed=42
)


cat_model.fit(model_train, label_train)

CatBoostClassifier(border_count=254, depth=6, iterations=500, l2_leaf_reg=3, learning_rate=0.1, random_seed=42, scale_pos_weight=np.float64(525.9825183683811), verbose=False)

In [44]:
probs = cat_model.predict_proba(model_test)[:, 1]

for threshold in [0.7, 0.8, 0.9, 0.95, 0.99]:
    preds = (probs >= threshold).astype(int)
    p = precision_score(label_test, preds)
    r = recall_score(label_test, preds)
    print(f"Threshold {threshold} | Precision: {p:.4f} | Recall: {r:.4f}")

Threshold 0.7 | Precision: 0.0151 | Recall: 0.1448
Threshold 0.8 | Precision: 0.0187 | Recall: 0.0947
Threshold 0.9 | Precision: 0.0196 | Recall: 0.0317
Threshold 0.95 | Precision: 0.0243 | Recall: 0.0099
Threshold 0.99 | Precision: 0.0000 | Recall: 0.0000


In [45]:
check_df = pd.DataFrame({"Bin": pd.qcut(probs, q=10, duplicates="drop"), "Label": label_test})
summary = check_df.groupby("Bin").agg({"Label": ["sum", "count"]})
summary = summary.sort_index(ascending=False)
summary

/tmp/ipykernel_84902/2412947802.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  summary = check_df.groupby("Bin").agg({"Label": ["sum", "count"]})


Label        
                                    sum   count
Bin                                            
(0.316, 0.993]                      772  104000
(0.143, 0.316]                      341  104000
(0.0577, 0.143]                     373  104000
(0.0159, 0.0577]                    262  104000
(0.00206, 0.0159]                   242  104000
(0.000205, 0.00206]                  78  103997
(3.85e-05, 0.000205]                 25  104002
(5.78e-06, 3.85e-05]                 17  104001
(3.85e-07, 5.78e-06]                  3  104000
(-0.000999999999999579, 3.85e-07]     0  104000

In [27]:
# #catboost
# cat_model = CatBoostClassifier(
#     iterations=100,           
#     learning_rate=0.1,
#     depth=6,                  
#     l2_leaf_reg=3,            
#     border_count=254,         
#     verbose=False,            
#     random_seed=42
# )

# # cat_model.fit(model_train, label_train)
# cat_model.fit(model_train, label_train)

In [46]:
y_pred_cat = cat_model.predict(model_test)

print(classification_report(label_test, y_pred_cat))

              precision    recall  f1-score   support

           0       1.00      0.95      0.97   1037887
           1       0.01      0.25      0.02      2113

    accuracy                           0.95   1040000
   macro avg       0.50      0.60      0.50   1040000
weighted avg       1.00      0.95      0.97   1040000

